# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show basic metadata info
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print("Keywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We will use the Croissant schema to enumerate metadata for the available record sets and their fields. Each will be referenced by its `@id`.

In [ ]:
# List record sets using their @id
record_sets = dataset.record_sets
print("Available Record Sets and Their Fields:")
record_set_ids = []
for rs in record_sets:
    print(f"- Record Set: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    print("  Fields:")
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) else f
        print(f"    - {f_id}")
    # Optionally, display columns
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for c in columns:
            c_id = c['@id'] if isinstance(c, dict) else c
            print(f"    - {c_id}")
    print()

# Explore a few records for each record set
for rs_id in record_set_ids:
    print(f"---- Sample Records from Record Set: {rs_id} ----")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if idx >= 2:
                break
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview for reference.

In [ ]:
# Extract data from record sets into DataFrames
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nDataFrame columns for Record Set {rs_id}: {df.columns.tolist()}")
            print(df.head(3))
        else:
            print(f"No records for {rs_id}.")
    except Exception as e:
        print(f"Could not extract records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will pick a numeric field from the primary record set and demonstrate filtering and normalization.

In [ ]:
# Choose a record set and numeric field to analyze
main_record_set_id = record_set_ids[0] if record_set_ids else None
df_main = dataframes.get(main_record_set_id)

# Find numeric columns
if df_main is not None:
    numeric_fields = [col for col in df_main.columns if df_main[col].dtype in ['int64','float64']]
    print(f"Numeric fields in {main_record_set_id}: {numeric_fields}")

    # Use 'phq9_score' as an example, fallback to first numeric field
    target_field_id = None
    for f in numeric_fields:
        if 'phq9' in f.lower() or 'score' in f.lower():
            target_field_id = f
            break
    if not target_field_id and numeric_fields:
        target_field_id = numeric_fields[0]

    # Filtering records
    threshold = 10
    if target_field_id:
        filtered_df = df_main[df_main[target_field_id] > threshold]
        print(f"Filtered records with {target_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        normalized_field = f"{target_field_id}_normalized"
        filtered_df[normalized_field] = (filtered_df[target_field_id] - filtered_df[target_field_id].mean()) / filtered_df[target_field_id].std()
        print(f"Normalized {target_field_id} for filtered records:")
        print(filtered_df[[target_field_id, normalized_field]].head())

        # Grouping by a categorical field (e.g. 'gender')
        group_field_candidates = [col for col in df_main.columns if col.lower() in ['gender','sex','village','level_of_education']]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for demonstration.")
else:
    print("No records found in main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll plot a histogram of the main numeric mental health score and a boxplot grouped by gender, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df_main is not None and target_field_id:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[target_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {target_field_id} in {main_record_set_id}")
    plt.xlabel(target_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    if group_field:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field, y=target_field_id, data=df_main)
        plt.title(f"{target_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(target_field_id)
        plt.show()
else:
    print("No main DataFrame or numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from dataset exploration.

- The dataset contains detailed survey records from Kilifi County, including demographic and mental health scores (e.g. PHQ-9, GAD-7).
- Using `mlcroissant`, we've loaded the dataset metadata and extracted record sets and fields referenced by their `@id`.
- Exploratory data analysis identified outlier cases (e.g., elevated PHQ-9 scores) and enabled normalization and subgroup analysis (e.g., by gender).
- Visualizations reveal distributions and group-wise differences, supporting further research into mental health patterns.

For more advanced analysis, consider cross-referencing multiple record sets and utilizing additional fields referenced by `@id`.